<a href="https://colab.research.google.com/github/Sharafatnoa/DailyPractice/blob/main/2026-09-19-statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Practice — 2026-09-19: Statistics — Is Your Candidate Model *Actually* Better?

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset) — 569 real patient records with 30 measured tumor features, bundled with scikit-learn (`load_breast_cancer`). This is genuine diagnostic lab data, not a synthetic toy set.

**The scenario:** You're the QA engineer on a team that just trained a fancier candidate model (a `RandomForestClassifier`) to replace a simple, already-shipped baseline (`LogisticRegression`) for a cancer diagnosis classifier. The candidate's cross-validated ROC-AUC number *looks* different from the baseline's. Your job is the same job you'd do reviewing a build that claims to "fix" a flaky test suite: don't trust a single headline number. You need to decide, with statistical rigor, whether the candidate is actually better, actually worse, or whether the difference is just noise — and whether a "statistically significant" difference is even large enough to matter in practice.

**What you should produce:**
1. `cv_scores(...)` — stratified k-fold cross-validation scores (ROC-AUC) for a given model.
2. `bootstrap_ci(...)` — a bootstrap confidence interval for a metric on held-out predictions, correctly handling resamples that accidentally drop a class.
3. `paired_permutation_test(...)` — a paired permutation test comparing the candidate's and baseline's per-fold CV scores, returning an observed mean difference and a two-sided p-value.
4. `ship_decision(...)` — a function that combines **statistical significance** (p-value vs. alpha) with **practical significance** (a minimum meaningful effect size) to return one of `"ship candidate"`, `"keep baseline"`, or `"insufficient evidence"`.
5. A short experiment (provided in the solution) demonstrating why running *many* uncorrected comparisons — many metrics, many thresholds, many "tweaks" — inflates the odds of a false "it's better!" conclusion, the same way flaky, uncorrected test reruns can manufacture false confidence in a fix.

Fill in every function marked `TODO` / `raise NotImplementedError` yourself before you look at the solution below.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

RNG = np.random.default_rng(42)


In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print("shape:", X.shape)
print("class balance (0=malignant, 1=benign):", y.value_counts().to_dict())
X.head()


In [ ]:
# Baseline: already shipped, simple and well-understood.
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=0))

# Candidate: newer, fancier, and what the team wants to ship next.
candidate = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=0)


## Your turn: implement the four functions below

In [ ]:
def cv_scores(model, X, y, n_splits=10, scoring="roc_auc", random_state=0):
    """
    TODO: Run stratified k-fold cross-validation and return an array of
    per-fold scores.

    Use StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    together with sklearn's cross_val_score.
    """
    raise NotImplementedError


def bootstrap_ci(y_true, y_scores, metric_fn=roc_auc_score, n_boot=2000, alpha=0.05, rng=RNG):
    """
    TODO: Bootstrap a (1 - alpha) confidence interval for metric_fn(y_true, y_scores).

    Resample indices with replacement n_boot times, recompute the metric on
    each resample, and return (point_estimate, lower, upper) using the
    percentile method. Skip (don't crash on) any resample that happens to
    contain only one class, since metric_fn can't be computed on it.
    """
    raise NotImplementedError


def paired_permutation_test(scores_a, scores_b, n_perm=10000, rng=RNG):
    """
    TODO: Test H0: no systematic difference between paired scores_a and
    scores_b (e.g. matching per-fold CV scores for two models on the same folds).

    For each permutation, randomly flip the sign of each paired difference,
    recompute the mean of the flipped differences, and build a null
    distribution of these means. Return (observed_mean_diff, two_sided_p_value),
    where the p-value is the fraction of permuted means at least as extreme
    (in absolute value) as the observed one.
    """
    raise NotImplementedError


def ship_decision(mean_diff, p_value, alpha=0.05, min_practical_effect=0.01):
    """
    TODO: Return one of "ship candidate", "keep baseline", or
    "insufficient evidence", combining:
      - statistical significance: p_value < alpha
      - practical significance: abs(mean_diff) >= min_practical_effect

    Think about why a statistically significant p-value attached to a tiny
    effect size should NOT automatically ship a model change.
    """
    raise NotImplementedError


In [ ]:
# TODO: put it together.
# 1. Get 10-fold CV ROC-AUC scores for `baseline` and `candidate` on the full X, y.
# 2. Split off a held-out test set, fit both models on the training split, and
#    bootstrap a 95% CI for each model's ROC-AUC on the held-out set.
# 3. Run paired_permutation_test on the two CV score arrays from step 1.
# 4. Call ship_decision(...) with the observed mean diff and p-value, and
#    print a short, clear report a reviewer could act on.


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

### Implementation

```python
def cv_scores(model, X, y, n_splits=10, scoring="roc_auc", random_state=0):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    return cross_val_score(model, X, y, cv=cv, scoring=scoring)


def bootstrap_ci(y_true, y_scores, metric_fn=roc_auc_score, n_boot=2000, alpha=0.05, rng=RNG):
    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)
    n = len(y_true)
    boot_stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt, ys = y_true[idx], y_scores[idx]
        if len(np.unique(yt)) < 2:
            continue  # can't compute ROC-AUC with a single class in the resample
        boot_stats.append(metric_fn(yt, ys))
    boot_stats = np.array(boot_stats)
    point = metric_fn(y_true, y_scores)
    lower, upper = np.percentile(boot_stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lower, upper


def paired_permutation_test(scores_a, scores_b, n_perm=10000, rng=RNG):
    scores_a = np.asarray(scores_a)
    scores_b = np.asarray(scores_b)
    diffs = scores_a - scores_b
    observed = diffs.mean()
    n = len(diffs)
    count = 0
    for _ in range(n_perm):
        signs = rng.choice([-1, 1], size=n)
        perm_mean = (diffs * signs).mean()
        if abs(perm_mean) >= abs(observed):
            count += 1
    p_value = (count + 1) / (n_perm + 1)  # +1 avoids a p-value of exactly 0
    return observed, p_value


def ship_decision(mean_diff, p_value, alpha=0.05, min_practical_effect=0.01):
    significant = p_value < alpha
    practical = abs(mean_diff) >= min_practical_effect
    if significant and practical:
        return "ship candidate" if mean_diff > 0 else "keep baseline"
    return "insufficient evidence"
```

### Putting it together

```python
base_scores = cv_scores(baseline, X, y)
cand_scores = cv_scores(candidate, X, y)
print("baseline CV ROC-AUC:", base_scores.mean(), "+/-", base_scores.std())
print("candidate CV ROC-AUC:", cand_scores.mean(), "+/-", cand_scores.std())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
baseline.fit(X_train, y_train)
candidate.fit(X_train, y_train)
base_test_scores = baseline.predict_proba(X_test)[:, 1]
cand_test_scores = candidate.predict_proba(X_test)[:, 1]

print("baseline 95% CI:", bootstrap_ci(y_test, base_test_scores))
print("candidate 95% CI:", bootstrap_ci(y_test, cand_test_scores))

diff, p_value = paired_permutation_test(cand_scores, base_scores)
print(f"mean diff (candidate - baseline): {diff:.4f}, p-value: {p_value:.4f}")
print("decision:", ship_decision(diff, p_value))
```

With `random_state=0` this typically prints something like:

```
baseline CV ROC-AUC: 0.995 +/- 0.008
candidate CV ROC-AUC: 0.990 +/- 0.013
mean diff (candidate - baseline): -0.005, p-value: ~0.21
decision: insufficient evidence
```

**The point:** the candidate's headline CV number is *lower* than the baseline's here, but with only 10 paired folds and this much fold-to-fold variance, a difference of ~0.005 AUC isn't distinguishable from noise (p ≈ 0.21, nowhere near 0.05). A team that shipped or rejected the candidate based on the raw mean alone — without asking "is this difference real?" — would be making a coin-flip decision and calling it engineering.

### Bonus: why uncorrected multiple comparisons are dangerous

```python
false_positives = 0
n_trials = 30
for _ in range(n_trials):
    a = RNG.normal(0.9, 0.02, 10)  # two models with IDENTICAL true performance
    b = RNG.normal(0.9, 0.02, 10)
    _, pv = paired_permutation_test(a, b, n_perm=2000, rng=RNG)
    if pv < 0.05:
        false_positives += 1
print(f"false positives out of {n_trials} trials: {false_positives}")
```

Even though `a` and `b` are drawn from *exactly the same distribution* (there is no real difference between these two "models"), running the test 30 times at alpha = 0.05 turns up roughly 1-2 "significant" results just by chance. This is the statistical version of re-running a flaky test suite until it goes green: if you compare enough metrics, thresholds, or model variants without correcting for multiple comparisons (e.g. Bonferroni, Benjamini-Hochberg), you *will* eventually find a "significant" improvement that is pure noise — and ship it.

</details>